In [ ]:
import sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")

In [ ]:
from b1k_pipeline.utils import PipelineFS, get_targets

In [ ]:
pipeline_fs = PipelineFS()

In [ ]:
import json
from tqdm.notebook import tqdm
from fs.zipfs import ZipFS

problematic_targets = []
for tgt in tqdm(get_targets("combined")):
    target_fs = pipeline_fs.target_output(tgt)
    assert target_fs.exists("object_list.json")
    mesh_list = json.loads(target_fs.readtext("object_list.json"))["meshes"]

    if not target_fs.exists("meshes.zip"):
        problematic_targets.append(tgt)
        print(tgt, "missing meshes file")
        continue

    missing_meshes = []
    with target_fs.open("meshes.zip", "rb") as zip_file, ZipFS(zip_file) as zip_fs:
        for mesh in mesh_list:
            if not zip_fs.exists(mesh):
                missing_meshes.append(mesh)

    if missing_meshes:
        problematic_targets.append(tgt)

In [ ]:
len(problematic_targets)

In [ ]:
problematic_targets

In [ ]:
print(
    "dvc repro -s -f", " ".join(f"export_meshes@{tgt}" for tgt in problematic_targets)
)

In [ ]:
import yaml, collections

with open(r"D:\BEHAVIOR-1K\asset_pipeline\dvc.lock", "r") as f:
    lock = yaml.load(f, Loader=yaml.SafeLoader)

In [ ]:
ctr = collections.Counter()
for stage_name, stage in lock["stages"].items():
    if not stage_name.startswith("export_meshes@"):
        continue

    (pyfile,) = [x for x in stage["deps"] if x["path"].endswith("export_meshes.py")]
    md5 = pyfile["md5"]
    size = pyfile["size"]
    ctr[(md5, size)] += 1
print(ctr.most_common())

In [ ]:
import hashlib

with open(r"D:\BEHAVIOR-1K\asset_pipeline\b1k_pipeline\max\export_meshes.py", "r") as f:
    print(hashlib.md5(f.read().encode("utf-8")).hexdigest())